# V2 FINAL LIVE DEMO — Colab launcher (Phase 21)

**Canonical viva launch vehicle** for the already-completed Streamlit artefact (`app/streamlit_app.py`).

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

This notebook does **not**:

- rerun the 420-case benchmark, calibration, LLM-as-judge, or statistics
- modify frozen 140/40, locked **T = 0.65**, Phase 15–18 results, V1, or RAG architectures
- fall back to mock or Ollama
- tell you to open `127.0.0.1:8501` on the Mac

Previous live notebooks (`colab_phase11_live.ipynb` and earlier) are **historical development/validation evidence**. Use **this** notebook for the MSc viva.

Requires the Phase 6 knowledge base on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/`.

## What the Streamlit app exposes (already implemented)

Pages: **Live RAG Demo** · **Benchmark Results** (read-only saved metrics) · **Benchmark Questions** (read-only frozen 140).

Live RAG Demo retains: fresh question; three independent architectures; retrieved evidence and scores/metadata; generated answer; Multi-Agent verification; UQ confidence; locked T=0.65; ANSWER/ABSTAIN; UI-only near-threshold warning; runtime/backend/GPU; ERROR/UNAVAILABLE handling.

## 1. Clone or pull the latest V2 from GitHub

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'cursor/empty-v2-workspace'
CLONE_DIR = Path('/content/capstone-rag')

if not Path('/content').exists():
    raise RuntimeError(
        'This notebook must run on Google Colab GPU. '
        'Do not launch the viva demo from the Mac.'
    )

def _git(*args: str) -> None:
    result = subprocess.run(['git', *args], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f'git {" ".join(args)} failed')

if (CLONE_DIR / '.git').is_dir():
    print('Existing clone — fetch/pull', BRANCH)
    _git('-C', str(CLONE_DIR), 'fetch', 'origin', BRANCH)
    _git('-C', str(CLONE_DIR), 'checkout', BRANCH)
    _git('-C', str(CLONE_DIR), 'pull', '--ff-only', 'origin', BRANCH)
else:
    print('Cloning branch:', BRANCH)
    _git('clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR))

V2_ROOT = CLONE_DIR / 'V2'
required = [
    V2_ROOT / 'app' / 'streamlit_app.py',
    V2_ROOT / 'results' / 'config' / 'threshold.lock.json',
    V2_ROOT / 'src' / 'models' / 'runtime_guard.py',
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(
        'Push the completed V2 live artefact to GitHub first. Missing: ' + ', '.join(missing)
    )

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Mount Google Drive and restore the Phase 6 knowledge base

Uses **only** `MyDrive/MSc-RAG/artifacts/knowledge_base/`. Does **not** rebuild the index and does **not** copy a Mac Chroma database.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_KB = Path('/content/drive/MyDrive/MSc-RAG/artifacts/knowledge_base')

for name in ('index', 'documents'):
    src = DRIVE_KB / name
    dst = V2 / 'knowledge_base' / name
    if not src.is_dir():
        raise FileNotFoundError(
            f'Phase 6 knowledge base missing on Drive: {src}. '
            'Restore MyDrive/MSc-RAG/artifacts/knowledge_base/ before the viva. '
            'Refusing to rebuild or fall back to mock.'
        )
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

print('Drive KB restore complete. Not running build_index.py.')

## 4. Verify Chroma index vs Phase 6 manifest

In [ ]:
from pathlib import Path
import json
import os
import sys

V2 = Path('/content/capstone-rag/V2')
os.chdir(V2)
sys.path.insert(0, str(V2))

from src.config import get_path, load_experiment_config, project_root
from src.retrieval.index import COLLECTION_NAME
from src.retrieval.preflight import validate_index_preflight

config = load_experiment_config()
retrieval_cfg = config.section('retrieval')
index_dir = get_path(config, 'kb_index')
manifest = (
    project_root()
    / str(retrieval_cfg.get('index_manifest') or 'knowledge_base/index/index_manifest.json')
).resolve()
preflight = validate_index_preflight(
    index_dir,
    manifest_path=manifest,
    collection_name=str(retrieval_cfg.get('collection_name') or COLLECTION_NAME),
)
n_chunks = int(preflight.get('actual_count') or 0)
expected = int(preflight.get('expected_chunks') or 0)
if n_chunks <= 0:
    raise RuntimeError('Chroma collection is empty. Restore the Phase 6 KB from Drive.')
if expected and n_chunks != expected:
    raise RuntimeError(
        f'Chunk count {n_chunks} does not match Phase 6 manifest expected_chunks={expected}.'
    )
print(json.dumps({
    'manifest': str(manifest),
    'actual_count': n_chunks,
    'expected_chunks': expected or n_chunks,
    'collection': str(retrieval_cfg.get('collection_name') or COLLECTION_NAME),
    'status': 'PASS',
}, indent=2))

## 5. Verify Colab CUDA, Tesla T4, Qwen3-8B Q4_K_M, llama_cpp

Refuses Darwin/Mac, CPU-only Colab, mock, and Ollama. Does **not** start a benchmark.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys

V2 = Path('/content/capstone-rag/V2')
if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError(
        'Refusing to launch: this is not a Google Colab CUDA GPU runtime. '
        'Open notebooks/colab_phase21_final_live_demo.ipynb on Colab with a T4 GPU. '
        'Do not start Streamlit on the Mac for the viva.'
    )

os.chdir(V2)
sys.path.insert(0, str(V2))
os.environ['V2_LIVE_BACKEND'] = 'llama_cpp'
os.environ['V2_FORBID_MOCK'] = '1'
os.environ['V2_REQUIRE_CUDA'] = '1'

import torch
from src.models.runtime_guard import verify_live_llama_cpp_runtime

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is not available. Runtime → Change runtime type → GPU (T4). '
        'Refusing mock and Ollama fallback.'
    )
gpu_name = torch.cuda.get_device_name(0)
if 'T4' not in gpu_name:
    raise RuntimeError(
        f'Refusing to launch: GPU is {gpu_name!r}, expected Tesla T4. '
        'No mock/Ollama fallback.'
    )

runtime = verify_live_llama_cpp_runtime(require_cuda=True)
gguf = str(runtime.get('gguf_filename') or '')
if 'Q4_K_M' not in gguf and 'Q4_K_M' not in str(runtime.get('gguf_path') or ''):
    raise RuntimeError(f'Qwen3-8B Q4_K_M GGUF not verified: {runtime}')
if runtime.get('backend') != 'llama_cpp' or runtime.get('device') != 'cuda':
    raise RuntimeError(f'Refusing to launch: {runtime}')

print(json.dumps(runtime, indent=2))
print('RUNTIME LOCKED: llama_cpp on', runtime.get('gpu'))
print('Mock and Ollama are forbidden.')

## 6. Viva banner (confirmed runtime)

In [ ]:
import torch

print('V2 FINAL LIVE DEMO')
print('Colab GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'UNAVAILABLE')
print('Backend: llama_cpp')
print('Research threshold: T=0.65 LOCKED')

## 7. Start Streamlit and print the Colab proxy URL

Launch command (inside this Colab VM, from `V2/`):

`PYTHONPATH=. python -m streamlit run app/streamlit_app.py --server.port=8501 --server.address=0.0.0.0 --server.headless=true`

The browser URL is produced by Colab `google.colab.kernel.proxyPort(8501)` (and an iframe via `serve_kernel_port_as_iframe`). Open **that** URL in the browser. Do **not** open `127.0.0.1:8501` on the Mac.

Does **not** run `run_full_benchmark.py`, `run_live_demo.py`, calibration, the judge, or statistics.

In [ ]:
import json
import os
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from google.colab import output
from google.colab.output import eval_js

V2 = Path('/content/capstone-rag/V2')
if not Path('/content').exists() or os.uname().sysname == 'Darwin':
    raise RuntimeError('Not on Colab CUDA. Refusing to start Streamlit (no mock/Ollama fallback).')
os.chdir(V2)

def _wait_port(port: int, timeout: int = 90) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        sock = socket.socket()
        try:
            sock.connect(('127.0.0.1', port))
            return True
        except OSError:
            time.sleep(1)
        finally:
            sock.close()
    return False

os.system("pkill -f 'streamlit run app/streamlit_app.py' || true")
time.sleep(2)

env = os.environ.copy()
env['PYTHONPATH'] = str(V2)
env['V2_LIVE_BACKEND'] = 'llama_cpp'
env['V2_FORBID_MOCK'] = '1'
env['V2_REQUIRE_CUDA'] = '1'
env['STREAMLIT_SERVER_ENABLE_CORS'] = 'false'
env['STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION'] = 'false'

st_log = Path('/tmp/v2_phase21_streamlit.log')
st_handle = st_log.open('w')

st_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', 'app/streamlit_app.py',
        '--server.port=8501',
        '--server.address=0.0.0.0',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
        '--browser.gatherUsageStats=false',
    ],
    cwd=str(V2),
    env=env,
    stdout=st_handle,
    stderr=subprocess.STDOUT,
)
print('Streamlit pid on Colab VM:', st_proc.pid)
if not _wait_port(8501):
    print(st_log.read_text()[-3000:])
    raise RuntimeError('Streamlit did not start inside this Colab runtime.')

health = urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=15)
print('Streamlit health:', health.status, health.read().decode('utf-8', errors='replace').strip())

url = eval_js('google.colab.kernel.proxyPort(8501)')
if not url:
    raise RuntimeError('Colab proxyPort returned no URL. Re-run this cell.')

Path('/tmp/v2_live_demo_url.txt').write_text(str(url) + '\n', encoding='utf-8')
print('=' * 72)
print('VIVA BROWSER URL — open this in the browser (Colab proxy, not the Mac):')
print(url)
print('=' * 72)
output.serve_kernel_port_as_iframe(8501, height=900)
print('Pages: Live RAG Demo · Benchmark Results · Benchmark Questions')
print('Backend locked to llama_cpp. Mock and Ollama are forbidden.')
print('Research threshold T=0.65 LOCKED. No benchmark is running.')

## 8. Optional: Streamlit process health check

Checks that the Streamlit HTTP server responds. Does **not** run RAG, Qwen, or any benchmark.

In [ ]:
import urllib.request
from pathlib import Path

url_file = Path('/tmp/v2_live_demo_url.txt')
proxy = url_file.read_text(encoding='utf-8').strip() if url_file.is_file() else None
health = urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=15)
body = health.read().decode('utf-8', errors='replace').strip()
print('optional_verification status=', health.status, 'body=', body)
print('proxy_url=', proxy)
if health.status != 200:
    raise RuntimeError('Streamlit health check failed.')
print('PASS — Streamlit process responded. No benchmark was executed.')

## 9. Keep Streamlit alive for the viva

Leave this cell running while you use the proxy URL. Interrupt only when the demo is finished.

In [ ]:
import socket
import time
from pathlib import Path

url_file = Path('/tmp/v2_live_demo_url.txt')
url = url_file.read_text(encoding='utf-8').strip() if url_file.is_file() else '(launch cell not run)'
print('VIVA BROWSER URL (Colab proxy):', url)
print('Keep this cell running. Interrupt when the live demo is done.')

while True:
    sock = socket.socket()
    try:
        sock.connect(('127.0.0.1', 8501))
        alive = True
    except OSError:
        alive = False
    finally:
        sock.close()
    if not alive:
        raise RuntimeError('Streamlit is no longer listening on the Colab VM. Re-run section 7.')
    time.sleep(30)